# TrustLens — Explanation Engine and Trust Calibration
### Day 4: SHAP Pipeline, Explanation Generators, and the Calibration Metric

---

**Project:** *TrustLens — Trust Calibration, Decision Agency, and Fairness Perception Across Explanation Modalities in AI-Assisted Loan Decision-Making*

This notebook builds the heart of the TrustLens system: the machinery that turns a model prediction into the three explanation styles participants will see, and the metric that lets us measure whether their trust is *well-placed* rather than merely high.

By the end of this notebook we will have:

1. A **SHAP explainer** that attributes each prediction to individual features — the engine behind the visual condition.
2. A **text explanation generator** (Condition B) that turns those attributions into plain English.
3. A **SHAP waterfall visual** (Condition C) that shows the same information graphically.
4. A small but important **decision helper** that guarantees every explanation matches the model's *real* output.
5. The **trust calibration scoring function** — the conceptual centrepiece of the whole study — validated on simulated participants before any real data is collected.

---

## 1. Setup

SHAP is not pre-installed in Colab, so we install it first, then import the libraries we will lean on throughout: pandas and numpy for data handling, matplotlib for plotting, joblib to reload the artefacts we saved during model training, and SHAP itself.

In [ ]:
!pip install shap -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import shap

## 2. Load the Model and Test Data

We reload everything we need from the model-training stage: the trained XGBoost classifier, the optimised decision threshold, and the held-out test set. Keeping the same threshold the model was evaluated with means the explanations in this notebook will always agree with the performance figures we reported earlier.

*When prompted, upload:* `xgboost_model.pkl`, `decision_threshold.pkl`, `X_test.csv`, `y_test.csv`.

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
model     = joblib.load('xgboost_model.pkl')
threshold = joblib.load('decision_threshold.pkl')
X_test    = pd.read_csv('X_test.csv')
y_test    = pd.read_csv('y_test.csv').squeeze()

Before building anything on top of these artefacts, we verify they loaded as expected. Checking the model type, the threshold value, and the data shapes takes a moment and saves a great deal of confusion later — a habit worth keeping in any research codebase.

In [ ]:
print('Model type     :', type(model).__name__)
print('Threshold      :', threshold)
print('X_test shape   :', X_test.shape)
print('y_test shape   :', y_test.shape)
print('Feature names  :', list(X_test.columns))

## 3. Build the SHAP Explainer

SHAP (SHapley Additive exPlanations) answers a very specific question for each applicant: *how much did each feature push this particular decision toward approval or rejection?* For tree-based models like XGBoost, `TreeExplainer` computes these contributions exactly and efficiently.

A helpful way to picture it: every prediction starts from a baseline (the model's average output across everyone), and each feature then nudges the prediction up or down until it lands on this person's final score. SHAP tells us the size and direction of every nudge.

We confirm success by checking that the SHAP values have the same shape as the test data — one contribution for every feature, for every applicant.

In [ ]:
explainer   = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

print('SHAP values shape :', shap_values.shape)
print('Base (expected) value :', round(float(explainer.expected_value), 3))
print('X_test shape          :', X_test.shape)

## 4. The Decision Helper — Always Use the Model's Real Output

This small function matters more than its size suggests. Every explanation a participant sees must correspond to the decision the model *actually* made — never to a value we typed in by hand. If the displayed decision and its explanation ever disagree, the study breaks: we cannot measure whether someone appropriately trusts a recommendation if the recommendation shown is not the real one.

`get_ai_decision` takes an applicant's index, asks the model for its probability of bad credit, applies our decision threshold, and returns both the decision and the confidence behind it. From here on, every decision flows from the model itself.

**A note on direction:** the model predicts the probability of *bad* credit. So a probability above the threshold means REJECT, and the confidence we report is the model's certainty in whichever direction it chose.

In [ ]:
def get_ai_decision(case_index):
    """Return the model's real decision and confidence for one applicant."""
    prob_bad   = model.predict_proba(X_test.iloc[[case_index]])[0, 1]
    decision   = 'REJECT' if prob_bad >= threshold else 'APPROVE'
    confidence = prob_bad if decision == 'REJECT' else 1 - prob_bad
    return decision, round(confidence, 3)

# Quick check on the first applicant
decision, confidence = get_ai_decision(0)
print(f'Applicant #0  ->  {decision}  ({confidence*100:.0f}% confident)')

## 5. Condition B — The Text Explanation Generator

The text condition translates the model's reasoning into a sentence a person can read without any technical background. We take the three features with the largest impact on this applicant, describe the strength of each (strongly, moderately, slightly), and state whether each one raised or lowered the assessed risk.

The strength thresholds (0.15 and 0.08) are deliberate design choices, calibrated so the language stays interpretable rather than cluttered. Noting such choices openly is part of doing this carefully — they can be revisited, and the study should acknowledge them.

Crucially, the decision passed into this function now comes from `get_ai_decision`, so the sentence always matches the model's real recommendation.

In [ ]:
def generate_text_explanation(shap_vals, feature_names, ai_decision):
    """Turn one applicant's SHAP values into a plain-English explanation (Condition B)."""
    # The three features with the largest absolute impact
    top_3_idx = np.argsort(np.abs(shap_vals))[::-1][:3]

    def strength(value):
        v = abs(value)
        if v >= 0.15:
            return 'strongly'
        elif v >= 0.08:
            return 'moderately'
        return 'slightly'

    reasons = []
    for idx in top_3_idx:
        feature   = feature_names[idx].replace('_', ' ')
        shap_v    = shap_vals[idx]
        direction = 'increased risk' if shap_v > 0 else 'reduced risk'
        reasons.append(f'{feature} {strength(shap_v)} {direction}')

    return (f'The AI recommends {ai_decision}. '
            f'The three main factors were: {reasons[0]}, {reasons[1]}, and {reasons[2]}.')


# Test it using the REAL decision
decision, confidence = get_ai_decision(0)
print(generate_text_explanation(shap_values[0], list(X_test.columns), decision))

## 6. Condition C — The SHAP Visual Generator

The visual condition presents the same reasoning as a waterfall plot. Reading from the bottom up, the chart starts at the model's baseline and shows each feature pushing the prediction left (toward approval, in blue) or right (toward rejection, in red) until it reaches the final score.

One point that is easy to misread: the horizontal axis is the model's internal risk score for *bad* credit, so more-negative values mean *less* risk. A long blue bar to the left is therefore a feature arguing strongly for approval. The visual makes trade-offs immediately obvious — including cases where a single strong factor points one way while several smaller factors collectively win out — which is precisely the kind of nuance the text condition cannot convey as cleanly.

In [ ]:
def generate_shap_visual(case_index, save_path=None):
    """Display a SHAP waterfall plot for one applicant (Condition C)."""
    explanation = shap.Explanation(
        values=shap_values[case_index],
        base_values=explainer.expected_value,
        data=X_test.iloc[case_index],
        feature_names=list(X_test.columns)
    )
    plt.figure()
    shap.plots.waterfall(explanation, show=False)
    decision, confidence = get_ai_decision(case_index)
    plt.title(f'Why the AI chose {decision} ({confidence*100:.0f}% confident) — Applicant #{case_index}',
              fontsize=12, fontweight='bold')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


generate_shap_visual(0)

## 7. The Trust Calibration Metric — The Heart of the Study

This is the idea that sets TrustLens apart from a typical explainability demo. Most studies ask whether explanations make people trust an AI *more*. We ask something harder and more useful: do explanations help people trust the AI *appropriately* — placing confidence in it when it is right, and withholding confidence when it is wrong?

To capture this, the metric reports three things rather than a single number:

- **Trust Calibration Score (TCS).** For each case we rescale the participant's trust rating to a 0–1 range and compare it to whether the AI was actually correct. The smaller the gap, the better calibrated the trust. TCS is one minus the average gap, so higher is better.
- **Overtrust index.** How often the participant placed *high* trust in a *wrong* decision — the dangerous failure mode, where a confident but incorrect AI is followed anyway.
- **Undertrust index.** How often the participant placed *low* trust in a *correct* decision — the wasteful failure mode, where sound advice is dismissed.

Reporting all three is what allows statements such as *"the visual explanation reduced overtrust without increasing undertrust"* — a level of nuance a single trust score simply cannot express.

The 0.5 cut-point used for the overtrust and undertrust indices is the midpoint of the rescaled trust scale; it is a transparent design choice and can be adjusted.

In [ ]:
def compute_trust_calibration(trust_ratings, ai_correctness, trust_scale_max=5):
    """
    Compute a participant's trust calibration from per-case trust ratings.

    trust_ratings   : list of Likert trust ratings (1..trust_scale_max), one per case
    ai_correctness  : list of 1 (AI correct) / 0 (AI wrong), one per case
    trust_scale_max : maximum value of the Likert scale (default 5)
    """
    trust   = np.array(trust_ratings, dtype=float)
    correct = np.array(ai_correctness, dtype=float)

    # 1. Rescale trust from [1..max] to [0..1]
    trust_scaled = (trust - 1) / (trust_scale_max - 1)

    # 2. Per-case calibration error
    calibration_errors = np.abs(trust_scaled - correct)

    # 3. Trust Calibration Score (higher = better)
    tcs = 1 - calibration_errors.mean()

    # 4. Overtrust: high trust on wrong decisions
    wrong_mask = (correct == 0)
    overtrust = (trust_scaled[wrong_mask] > 0.5).mean() if wrong_mask.sum() > 0 else 0.0

    # 5. Undertrust: low trust on correct decisions
    correct_mask = (correct == 1)
    undertrust = (trust_scaled[correct_mask] < 0.5).mean() if correct_mask.sum() > 0 else 0.0

    return {
        'TCS': round(tcs, 3),
        'overtrust_index': round(overtrust, 3),
        'undertrust_index': round(undertrust, 3),
        'calibration_errors': calibration_errors.round(3).tolist()
    }

## 8. Validating the Metric Before Any Real Data

A measure is only trustworthy if we know it behaves correctly. Before a single participant takes part, we test the metric against four simulated response patterns whose "correct" answers we already know: someone well-calibrated, someone who overtrusts, someone who undertrusts, and someone indifferent who rates everything the same.

If the metric is sound, the well-calibrated profile should score high, the overtruster should light up the overtrust index, the undertruster the undertrust index, and the indifferent participant should sit near the midpoint. Confirming this now means that when real data arrives, we can be confident the numbers mean what we claim they mean.

The six-case correctness pattern below — correct, correct, wrong, wrong, correct, wrong — mirrors the study design we will finalise next.

In [ ]:
ai_correct = [1, 1, 0, 0, 1, 0]   # correctness pattern of the six study cases

simulated = {
    'Well-calibrated': [5, 4, 1, 2, 5, 1],
    'Overtrusting':    [5, 5, 5, 4, 5, 5],
    'Undertrusting':   [2, 1, 1, 2, 2, 1],
    'Indifferent':     [3, 3, 3, 3, 3, 3],
}

for name, ratings in simulated.items():
    r = compute_trust_calibration(ratings, ai_correct)
    print(f"{name:16s} -> TCS={r['TCS']}, overtrust={r['overtrust_index']}, undertrust={r['undertrust_index']}")

### Reading the validation output

Two results are worth pausing on. The overtrusting and undertrusting participants land on the *same* TCS, yet the overtrust and undertrust indices separate them cleanly — one fails by trusting a wrong AI, the other by doubting a correct one. A single trust score would have hidden that distinction entirely. This is the concrete payoff of a multi-component metric, and the reason it can support the kind of nuanced claim a reviewer takes seriously.

## 9. Save the Pipeline Artefacts

Finally we persist the SHAP values and base value so later notebooks (case selection, analysis) can reuse them without recomputing. The explanation functions and the calibration metric live in this notebook for now and will be lifted into the application code during the build phase.

In [ ]:
joblib.dump(shap_values, 'shap_values.pkl')
joblib.dump(float(explainer.expected_value), 'shap_base_value.pkl')

from google.colab import files
for f in ['shap_values.pkl', 'shap_base_value.pkl']:
    files.download(f)
print('Saved and downloaded: shap_values.pkl, shap_base_value.pkl')

## Summary

This notebook assembled the explanation engine and the measurement core of TrustLens. We built a SHAP explainer over the test set, created the text and visual explanation generators that realise Conditions B and C, and introduced a decision helper that keeps every explanation faithful to the model's real output. Most importantly, we defined a three-part trust calibration metric — calibration score, overtrust, and undertrust — and validated it against simulated participants so its behaviour is understood before any real data is gathered.

**Artefacts to file in the repository**
- `shap_values.pkl`, `shap_base_value.pkl` -> `models/`
- this notebook -> `notebooks/03_shap_pipeline.ipynb`

**Version control**
```bash
git add . && git commit -m "Day 4: SHAP pipeline, explanation generators, trust calibration metric" && git push
```

**Next — Day 5:** selecting the six study cases using the crossed correctness x confidence design, layered with fairness-proxy cases, drawn directly from the model's real predictions.